# NASA Battery Tableau CSV Builder

이 노트북은 **`03_ml_Rct_last.ipynb`의 전처리/ML 기준을 유지**하면서,  
**Tableau 작업용 CSV를 한 번에 생성**하는 목적의 실무용 노트북입니다.

## 이 노트북이 만드는 것
- Overview 탭용 CSV
- 이상탐지(Anomaly) 탭용 CSV
- ML 탭용 CSV
- raw 파일 연결 여부 점검용 CSV
- 전체 export 파일 목록(manifest)
- CSV 묶음 ZIP 파일

## 기본 원칙
- core summary는 `metadata.csv`만으로도 생성 가능
- raw 상세 곡선은 raw csv 파일이 있을 때만 생성
- raw 파일이 부족해도 노트북은 **오류 없이 끝까지 실행**되도록 설계
- ML 결과는 **main / supervised** 기준 배터리만 사용


## 0. 경로/환경 설정

- 아래 셀은 `metadata.csv`, 원본 노트북, raw csv를 자동으로 찾습니다.
- 사용자가 폴더 구조를 조금 다르게 두어도 최대한 자동 탐색하도록 만들었습니다.
- 그래도 못 찾으면, `metadata_path`만 직접 지정해도 됩니다.


In [1]:

from pathlib import Path
import os
import re
import json
import math
import shutil
import warnings
import zipfile

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, LeaveOneGroupOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path('/mnt/data'),
    Path.cwd() / 'data',
    Path('/mnt/data/data')
]

def unique_existing_paths(path_list):
    seen = set()
    out = []
    for p in path_list:
        try:
            p = Path(p).resolve()
        except Exception:
            p = Path(p)
        if p.exists():
            key = str(p)
            if key not in seen:
                seen.add(key)
                out.append(p)
    return out

PROJECT_ROOT_CANDIDATES = unique_existing_paths(PROJECT_ROOT_CANDIDATES)
PROJECT_ROOT_CANDIDATES


[WindowsPath('C:/Users/박정수/OneDrive/바탕 화면/spartal_python/16_project/project_lithium_battery_analysis_16team'),
 WindowsPath('C:/Users/박정수/OneDrive/바탕 화면/spartal_python/16_project/project_lithium_battery_analysis_16team/data')]

## 1. 공통 함수

이 섹션은 전처리, 파일 탐색, 회귀모델 학습에 필요한 공통 함수입니다.  
원본 `03_ml_Rct_last.ipynb`의 핵심 규칙을 그대로 가져오되, Tableau export용으로 정리했습니다.


In [2]:

def find_first_existing_file(file_candidates):
    for one_path in file_candidates:
        one_path = Path(one_path)
        if one_path.exists():
            return one_path
    return None

def clear_time(x):
    if pd.isna(x):
        return pd.NaT

    x = str(x)
    nums = []
    current = ''

    for ch in x:
        if ch.isdigit() or ch in ['.', 'e', 'E', '+', '-']:
            current += ch
        else:
            if current != '':
                nums.append(current)
                current = ''

    if current != '':
        nums.append(current)

    if len(nums) < 6:
        return pd.NaT

    try:
        nums = list(map(float, nums[:6]))
    except Exception:
        return pd.NaT

    year, month, day, hour, minute, second = nums

    if not (
        2000 <= year <= 2100 and
        1 <= month <= 12 and
        1 <= day <= 31 and
        0 <= hour < 24 and
        0 <= minute < 60 and
        0 <= second < 60
    ):
        return pd.NaT

    try:
        return pd.Timestamp(
            int(year), int(month), int(day),
            int(hour), int(minute), int(second)
        )
    except Exception:
        return pd.NaT

def classify_capacity(cap):
    if pd.isna(cap):
        return 'missing'
    if cap == 0:
        return 'zero'
    if cap > 2.1:
        return 'impossible_high'
    if cap < 0.3:
        return 'low_anomaly'
    return 'valid'

def classify_impedance(row):
    re_value = row['Re']
    rct_value = row['Rct']
    diff_value = row['re_diff']

    if pd.isna(re_value) or pd.isna(rct_value):
        return 'missing'
    if re_value <= 0 or rct_value <= 0:
        return 'zero_or_minus'
    if pd.notna(diff_value) and diff_value > 0.3:
        return 'noise_candidate'
    if rct_value > re_value * 10:
        return 'rct_imbalance'
    if re_value > 1.0 or rct_value > 1.5:
        return 'high_anomaly'
    return 'valid'

def classify_return_status(row):
    threshold = row['eol_soh_threshold']
    soh_value = row['SOH_nominal']

    if pd.isna(threshold) or pd.isna(soh_value):
        return '판정보류'
    if soh_value < threshold:
        return '귀환 불가'
    if soh_value < threshold + 5:
        return '경고'
    return '귀환 가능'

def make_xgb_regressor(random_state=42):
    return XGBRegressor(
        objective='reg:squarederror',
        n_estimators=150,
        max_depth=2,
        learning_rate=0.05,
        subsample=1.0,
        colsample_bytree=1.0,
        random_state=random_state,
        n_jobs=1
    )

def fill_missing_with_train_median(X_train, X_test, feature_cols):
    fill_values = {}
    X_train_filled = X_train.copy()
    X_test_filled = X_test.copy()

    for col in feature_cols:
        fill_value = pd.to_numeric(X_train_filled[col], errors='coerce').median()
        if pd.isna(fill_value):
            fill_value = 0

        fill_values[col] = fill_value
        X_train_filled[col] = pd.to_numeric(X_train_filled[col], errors='coerce').fillna(fill_value)
        X_test_filled[col] = pd.to_numeric(X_test_filled[col], errors='coerce').fillna(fill_value)

    return X_train_filled, X_test_filled, fill_values

def prepare_model_table(model_df, feature_cols, target_col, group_col='battery_id'):
    work_df = model_df.copy()

    if 'start_time' in work_df.columns:
        work_df['start_time'] = pd.to_datetime(work_df['start_time'], errors='coerce')

    for col in feature_cols + [target_col]:
        work_df[col] = pd.to_numeric(work_df[col], errors='coerce')

    if 'impedance_available' in work_df.columns:
        work_df['impedance_available'] = work_df['impedance_available'].fillna(0).astype(int)

    work_df = work_df[work_df[group_col].notna()].copy()
    work_df = work_df[work_df[target_col].notna()].copy()

    sort_cols = [group_col]
    if 'start_time' in work_df.columns:
        sort_cols.append('start_time')

    work_df = work_df.sort_values(sort_cols).reset_index(drop=True)
    return work_df

def run_single_split_regression(model_df, feature_cols, target_col, model_factory, group_col='battery_id', random_state=42, test_size=0.25):
    work_df = prepare_model_table(model_df, feature_cols, target_col, group_col=group_col)

    if len(work_df) == 0:
        return {'status': 'skipped', 'reason': '학습 가능한 행이 없습니다.'}

    if work_df[group_col].nunique() < 2:
        return {'status': 'skipped', 'reason': '배터리가 최소 2개 이상 필요합니다.'}

    run_X = work_df[feature_cols].copy()
    run_y = work_df[target_col].copy()
    run_groups = work_df[group_col].copy()

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(run_X, run_y, run_groups))

    X_train = run_X.iloc[train_idx].copy()
    X_test = run_X.iloc[test_idx].copy()
    y_train = run_y.iloc[train_idx].copy()
    y_test = run_y.iloc[test_idx].copy()

    train_meta = work_df.iloc[train_idx].copy()
    test_meta = work_df.iloc[test_idx].copy()

    X_train_filled, X_test_filled, fill_values = fill_missing_with_train_median(
        X_train=X_train,
        X_test=X_test,
        feature_cols=feature_cols
    )

    if model_factory == LinearRegression:
        model = LinearRegression()
    else:
        model = model_factory()

    model.fit(X_train_filled, y_train)

    train_pred = model.predict(X_train_filled)
    test_pred = model.predict(X_test_filled)

    metric_df = pd.DataFrame([
        ['train', mean_absolute_error(y_train, train_pred), mean_squared_error(y_train, train_pred) ** 0.5, r2_score(y_train, train_pred)],
        ['test',  mean_absolute_error(y_test,  test_pred),  mean_squared_error(y_test,  test_pred)  ** 0.5, r2_score(y_test,  test_pred)]
    ], columns=['set', 'MAE', 'RMSE', 'R2']).round(4)

    pred_train_df = train_meta.copy()
    pred_train_df['set'] = 'train'
    pred_train_df['y_true'] = y_train.values
    pred_train_df['y_pred'] = train_pred

    pred_test_df = test_meta.copy()
    pred_test_df['set'] = 'test'
    pred_test_df['y_true'] = y_test.values
    pred_test_df['y_pred'] = test_pred

    pred_df = pd.concat([pred_train_df, pred_test_df], ignore_index=True)
    pred_df['abs_error'] = (pred_df['y_true'] - pred_df['y_pred']).abs()
    pred_df['error_band'] = pd.cut(
        pred_df['abs_error'],
        bins=[-np.inf, 5, 10, 20, np.inf],
        labels=['0-5', '5-10', '10-20', '20+']
    )

    out = {
        'status': 'ok',
        'reason': None,
        'feature_cols': feature_cols,
        'target_col': target_col,
        'metric_df': metric_df,
        'pred_df': pred_df,
        'fill_values': fill_values,
        'model': model
    }

    if hasattr(model, 'feature_importances_'):
        out['importance_df'] = pd.DataFrame({
            'feature': feature_cols,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False).reset_index(drop=True)
    else:
        out['importance_df'] = None

    if hasattr(model, 'coef_'):
        out['coef_df'] = pd.DataFrame({
            'feature': feature_cols,
            'coef': model.coef_
        }).sort_values('coef', key=np.abs, ascending=False).reset_index(drop=True)
    else:
        out['coef_df'] = None

    return out

def fit_full_model(model_df, feature_cols, target_col, model_factory, group_col='battery_id'):
    work_df = prepare_model_table(model_df, feature_cols, target_col, group_col=group_col)

    run_X = work_df[feature_cols].copy()
    run_y = work_df[target_col].copy()

    fill_values = {}
    run_X_filled = run_X.copy()

    for col in feature_cols:
        fill_value = pd.to_numeric(run_X_filled[col], errors='coerce').median()
        if pd.isna(fill_value):
            fill_value = 0

        fill_values[col] = fill_value
        run_X_filled[col] = pd.to_numeric(run_X_filled[col], errors='coerce').fillna(fill_value)

    if model_factory == LinearRegression:
        model = LinearRegression()
    else:
        model = model_factory()

    model.fit(run_X_filled, run_y)
    full_pred = model.predict(run_X_filled)

    pred_df = work_df.copy()
    pred_df['y_true'] = run_y.values
    pred_df['y_pred'] = full_pred
    pred_df['abs_error'] = (pred_df['y_true'] - pred_df['y_pred']).abs()

    out = {
        'model': model,
        'fill_values': fill_values,
        'pred_df': pred_df
    }

    if hasattr(model, 'feature_importances_'):
        out['importance_df'] = pd.DataFrame({
            'feature': feature_cols,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False).reset_index(drop=True)
    else:
        out['importance_df'] = None

    return out

def run_lobo(model_df, feature_cols, target_col, model_factory, group_col='battery_id'):
    work_df = prepare_model_table(model_df, feature_cols, target_col, group_col=group_col)

    if work_df[group_col].nunique() < 2:
        return pd.DataFrame(), pd.DataFrame()

    run_X = work_df[feature_cols].copy()
    run_y = work_df[target_col].copy()
    run_groups = work_df[group_col].copy()

    logo = LeaveOneGroupOut()

    detail_rows = []
    pred_parts = []

    for train_idx, test_idx in logo.split(run_X, run_y, run_groups):
        X_train = run_X.iloc[train_idx].copy()
        X_test = run_X.iloc[test_idx].copy()
        y_train = run_y.iloc[train_idx].copy()
        y_test = run_y.iloc[test_idx].copy()
        test_meta = work_df.iloc[test_idx].copy()

        X_train_filled, X_test_filled, fill_values = fill_missing_with_train_median(
            X_train=X_train,
            X_test=X_test,
            feature_cols=feature_cols
        )

        if model_factory == LinearRegression:
            model = LinearRegression()
        else:
            model = model_factory()

        model.fit(X_train_filled, y_train)
        test_pred = model.predict(X_test_filled)

        held_out_battery = test_meta[group_col].iloc[0]

        detail_rows.append({
            'held_out_battery': held_out_battery,
            'MAE': mean_absolute_error(y_test, test_pred),
            'RMSE': mean_squared_error(y_test, test_pred) ** 0.5,
            'R2': r2_score(y_test, test_pred),
            'row_count': len(test_meta)
        })

        one_pred_df = test_meta.copy()
        one_pred_df['held_out_battery'] = held_out_battery
        one_pred_df['y_true'] = y_test.values
        one_pred_df['y_pred'] = test_pred
        one_pred_df['abs_error'] = (one_pred_df['y_true'] - one_pred_df['y_pred']).abs()
        pred_parts.append(one_pred_df)

    detail_df = pd.DataFrame(detail_rows).round(4)
    pred_df = pd.concat(pred_parts, ignore_index=True) if len(pred_parts) > 0 else pd.DataFrame()

    return detail_df, pred_df


## 2. 메타데이터 로드

이 셀에서 `metadata.csv`를 찾습니다.  
기본적으로 아래 순서대로 찾습니다.

1. 현재 작업 폴더
2. `/mnt/data`
3. `data/metadata.csv`
4. `/mnt/data/data/metadata.csv`


In [3]:

metadata_candidates = [
    Path('metadata.csv'),
    Path('/mnt/data/metadata.csv'),
    Path('data/metadata.csv'),
    Path('/mnt/data/data/metadata.csv')
]

metadata_path = find_first_existing_file(metadata_candidates)

if metadata_path is None:
    raise FileNotFoundError('metadata.csv를 찾지 못했습니다. 경로를 확인해주세요.')

original_notebook_candidates = [
    Path('03_ml_Rct_last.ipynb'),
    Path('/mnt/data/03_ml_Rct_last.ipynb'),
]

original_notebook_path = find_first_existing_file(original_notebook_candidates)

print('metadata_path         :', metadata_path)
print('original_notebook_path:', original_notebook_path)

df = pd.read_csv(metadata_path)
print('원본 shape:', df.shape)
df.head()


metadata_path         : data\metadata.csv
original_notebook_path: 03_ml_Rct_last.ipynb
원본 shape: (7565, 10)


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN


## 3. 그룹/실험 조건 매핑

`03_ml_Rct_last.ipynb`에서 사용한 그룹 구조를 그대로 씁니다.

- A: main
- C: comparison
- F/G/H/I: anomaly
- B/D/E: excluded


In [4]:

group_map = {
    'B0005':'A','B0006':'A','B0007':'A','B0018':'A',
    'B0025':'B','B0026':'B','B0027':'B','B0028':'B',
    'B0029':'C','B0030':'C','B0031':'C','B0032':'C',
    'B0033':'D','B0034':'D','B0036':'D',
    'B0038':'E','B0039':'E','B0040':'E',
    'B0041':'F','B0042':'F','B0043':'F','B0044':'F',
    'B0045':'G','B0046':'G','B0047':'G','B0048':'G',
    'B0049':'H','B0050':'H','B0051':'H','B0052':'H',
    'B0053':'I','B0054':'I','B0055':'I','B0056':'I',
}

end_reason_map = {
    'A': 'EOL',      'B': 'censored', 'C': 'censored',
    'D': 'QA_issue', 'E': 'QA_issue', 'F': 'QA_issue',
    'G': 'QA_issue', 'H': 'crashed',  'I': 'QA_issue',
}

analysis_role_map = {
    'A': 'main',    'B': 'excluded',  'C': 'comparison',
    'D': 'excluded','E': 'excluded',  'F': 'anomaly',
    'G': 'anomaly', 'H': 'anomaly',   'I': 'anomaly',
}

test_temperature_profile_map = {
    'A': '24°C_stable',    'B': '24°C_stable',
    'C': '43°C_stable',    'D': '24°C_stable',
    'E': '24_44°C_mixed',
    'F': '4°C_22°C_mixed',
    'G': '4°C_stable',    'H': '4°C_stable',    'I': '4°C_stable',
}

load_profile_map = {
    'A': '2A_CC',         'B': '4A_squarewave', 'C': '4A_CC',
    'D': '2A_4A_mixed',   'E': '1A_2A_4A_mixed','F': '4A_1A_mixed',
    'G': '1A_CC',         'H': '2A_CC',         'I': '2A_CC',
}

eol_rule_source_map = {
    'A': 'NASA_30%fade', 'B': 'censored',      'C': 'censored',
    'D': 'NASA_20%fade', 'E': 'NASA_20%fade',
    'F': 'NASA_30%fade', 'G': 'NASA_30%fade',
    'H': 'crashed',      'I': 'NASA_30%fade',
}

battery_protocol_map = {
    'B0005': {'cutoff_voltage': 2.7, 'discharge_current': '2A_CC',          'eol_fade': 0.30},
    'B0006': {'cutoff_voltage': 2.5, 'discharge_current': '2A_CC',          'eol_fade': 0.30},
    'B0007': {'cutoff_voltage': 2.2, 'discharge_current': '2A_CC',          'eol_fade': 0.30},
    'B0018': {'cutoff_voltage': 2.5, 'discharge_current': '2A_CC',          'eol_fade': 0.30},
    'B0025': {'cutoff_voltage': 2.0, 'discharge_current': '4A_squarewave',  'eol_fade': None},
    'B0026': {'cutoff_voltage': 2.2, 'discharge_current': '4A_squarewave',  'eol_fade': None},
    'B0027': {'cutoff_voltage': 2.5, 'discharge_current': '4A_squarewave',  'eol_fade': None},
    'B0028': {'cutoff_voltage': 2.7, 'discharge_current': '4A_squarewave',  'eol_fade': None},
    'B0029': {'cutoff_voltage': 2.0, 'discharge_current': '4A_CC',          'eol_fade': None},
    'B0030': {'cutoff_voltage': 2.2, 'discharge_current': '4A_CC',          'eol_fade': None},
    'B0031': {'cutoff_voltage': 2.5, 'discharge_current': '4A_CC',          'eol_fade': None},
    'B0032': {'cutoff_voltage': 2.7, 'discharge_current': '4A_CC',          'eol_fade': None},
    'B0033': {'cutoff_voltage': 2.0, 'discharge_current': '4A_CC',          'eol_fade': 0.20},
    'B0034': {'cutoff_voltage': 2.2, 'discharge_current': '4A_CC',          'eol_fade': 0.20},
    'B0036': {'cutoff_voltage': 2.7, 'discharge_current': '2A_CC',          'eol_fade': 0.20},
    'B0038': {'cutoff_voltage': 2.2, 'discharge_current': '1A_2A_4A_mixed', 'eol_fade': 0.20},
    'B0039': {'cutoff_voltage': 2.5, 'discharge_current': '1A_2A_4A_mixed', 'eol_fade': 0.20},
    'B0040': {'cutoff_voltage': 2.7, 'discharge_current': '1A_2A_4A_mixed', 'eol_fade': 0.20},
    'B0041': {'cutoff_voltage': 2.0, 'discharge_current': '4A_1A_mixed',    'eol_fade': 0.30},
    'B0042': {'cutoff_voltage': 2.2, 'discharge_current': '4A_1A_mixed',    'eol_fade': 0.30},
    'B0043': {'cutoff_voltage': 2.5, 'discharge_current': '4A_1A_mixed',    'eol_fade': 0.30},
    'B0044': {'cutoff_voltage': 2.7, 'discharge_current': '4A_1A_mixed',    'eol_fade': 0.30},
    'B0045': {'cutoff_voltage': 2.0, 'discharge_current': '1A_CC',          'eol_fade': 0.30},
    'B0046': {'cutoff_voltage': 2.2, 'discharge_current': '1A_CC',          'eol_fade': 0.30},
    'B0047': {'cutoff_voltage': 2.5, 'discharge_current': '1A_CC',          'eol_fade': 0.30},
    'B0048': {'cutoff_voltage': 2.7, 'discharge_current': '1A_CC',          'eol_fade': 0.30},
    'B0049': {'cutoff_voltage': 2.0, 'discharge_current': '2A_CC',          'eol_fade': None},
    'B0050': {'cutoff_voltage': 2.2, 'discharge_current': '2A_CC',          'eol_fade': None},
    'B0051': {'cutoff_voltage': 2.5, 'discharge_current': '2A_CC',          'eol_fade': None},
    'B0052': {'cutoff_voltage': 2.7, 'discharge_current': '2A_CC',          'eol_fade': None},
    'B0053': {'cutoff_voltage': 2.0, 'discharge_current': '2A_CC',          'eol_fade': 0.30},
    'B0054': {'cutoff_voltage': 2.2, 'discharge_current': '2A_CC',          'eol_fade': 0.30},
    'B0055': {'cutoff_voltage': 2.5, 'discharge_current': '2A_CC',          'eol_fade': 0.30},
    'B0056': {'cutoff_voltage': 2.7, 'discharge_current': '2A_CC',          'eol_fade': 0.30},
}

eol_threshold_map = {
    'A': 70, 'B': 70, 'C': 70,
    'D': 80, 'E': 80,
    'F': 70, 'G': 70, 'H': None, 'I': 70
}


## 4. 전처리

여기서 하는 일은 아래와 같습니다.

1. `start_time` 정리  
2. type별 테이블 분리  
3. Capacity / Impedance flag 생성  
4. discharge cycle 번호 생성  
5. SOH / EOL / RUL 계산  
6. discharge row에 가장 최근 impedance(`Rct_last`, `Re_last`)를 붙여서 Tableau용 cycle master 생성


In [5]:

df['start_time'] = df['start_time'].apply(clear_time)
df = df.dropna(subset=['start_time']).copy()

df['group'] = df['battery_id'].map(group_map)
df['end_reason'] = df['group'].map(end_reason_map)
df['analysis_role'] = df['group'].map(analysis_role_map)
df['test_temperature_profile'] = df['group'].map(test_temperature_profile_map)
df['load_profile'] = df['group'].map(load_profile_map)
df['eol_rule_source'] = df['group'].map(eol_rule_source_map)

df['cutoff_voltage'] = df['battery_id'].map({k: v['cutoff_voltage'] for k, v in battery_protocol_map.items()})
df['discharge_current'] = df['battery_id'].map({k: v['discharge_current'] for k, v in battery_protocol_map.items()})
df['eol_fade_threshold'] = df['battery_id'].map({k: v['eol_fade'] for k, v in battery_protocol_map.items()})

df_charge = df[df['type'] == 'charge'].copy()
df_discharge = df[df['type'] == 'discharge'].copy()
df_impedance = df[df['type'] == 'impedance'].copy()

df_discharge['Capacity'] = pd.to_numeric(df_discharge['Capacity'], errors='coerce')
df_discharge['cap_flag'] = df_discharge['Capacity'].apply(classify_capacity)
df_discharge['cap_exclude'] = df_discharge['cap_flag'].isin(['missing', 'zero', 'impossible_high'])
df_discharge['cap_anomaly'] = df_discharge['cap_flag'].eq('low_anomaly')

df_impedance['Re'] = pd.to_numeric(df_impedance['Re'], errors='coerce')
df_impedance['Rct'] = pd.to_numeric(df_impedance['Rct'], errors='coerce')
df_impedance = df_impedance.sort_values(['battery_id', 'start_time']).reset_index(drop=True)
df_impedance['re_diff'] = df_impedance.groupby('battery_id')['Re'].diff().abs()
df_impedance['imp_flag'] = df_impedance.apply(classify_impedance, axis=1)
df_impedance['imp_exclude'] = df_impedance['imp_flag'].isin(['missing', 'zero_or_minus'])
df_impedance['imp_anomaly'] = df_impedance['imp_flag'].isin(['high_anomaly', 'noise_candidate', 'rct_imbalance'])
df_impedance['battery_impedance'] = df_impedance['Re'] + df_impedance['Rct']

df_discharge = df_discharge.sort_values(['battery_id', 'start_time', 'filename']).reset_index(drop=True)
df_discharge['discharge_cycle_raw'] = df_discharge.groupby('battery_id').cumcount() + 1
df_discharge['discharge_cycle_valid'] = (
    df_discharge.loc[~df_discharge['cap_exclude']]
    .groupby('battery_id')
    .cumcount() + 1
)

init_cap = (
    df_discharge[df_discharge['cap_flag'] == 'valid']
    .groupby('battery_id')['Capacity']
    .apply(lambda s: s.head(5).median())
    .rename('init_cap')
)

df_discharge = df_discharge.merge(init_cap, on='battery_id', how='left')

df_discharge['SOH_nominal'] = np.where(
    df_discharge['cap_flag'] == 'valid',
    (df_discharge['Capacity'] / 2.0 * 100).round(2),
    np.nan
)

df_discharge['SOH_relative'] = np.where(
    df_discharge['cap_flag'] == 'valid',
    (df_discharge['Capacity'] / df_discharge['init_cap'] * 100).round(2),
    np.nan
)

df_discharge['eol_soh_threshold'] = df_discharge['group'].map(eol_threshold_map)

eol_cycles = (
    df_discharge[
        (df_discharge['cap_flag'] == 'valid') &
        (df_discharge['eol_soh_threshold'].notna()) &
        (df_discharge['SOH_nominal'] < df_discharge['eol_soh_threshold'])
    ]
    .groupby('battery_id')['discharge_cycle_raw']
    .min()
    .rename('eol_cycle')
    .reset_index()
)

df_discharge = df_discharge.merge(eol_cycles, on='battery_id', how='left')

df_discharge['RUL'] = np.where(
    df_discharge['eol_cycle'].notna(),
    (df_discharge['eol_cycle'] - df_discharge['discharge_cycle_raw']).clip(lower=0),
    np.nan
)

df_discharge['rul_label_type'] = np.where(
    df_discharge['analysis_role'] == 'main', 'supervised',
    np.where(
        df_discharge['analysis_role'] == 'comparison', 'censored',
        np.where(
            df_discharge['analysis_role'] == 'anomaly', 'anomaly_case', 'unsupported_for_rul'
        )
    )
)

df_impedance['impedance_cycle_no'] = df_impedance.groupby('battery_id').cumcount() + 1

imp_valid = df_impedance[~df_impedance['imp_exclude']].copy()
imp_valid = imp_valid.sort_values(['battery_id', 'start_time']).reset_index(drop=True)

aof_rows = []

for battery_id in df_discharge['battery_id'].dropna().unique():
    one_dis = df_discharge[df_discharge['battery_id'] == battery_id].sort_values('start_time').copy()
    one_imp = imp_valid[imp_valid['battery_id'] == battery_id][
        ['start_time', 'Re', 'Rct', 'battery_impedance', 'imp_flag', 'imp_anomaly', 'impedance_cycle_no']
    ].sort_values('start_time').copy()

    if len(one_imp) == 0:
        merged = one_dis.copy()
        merged['Re_last'] = np.nan
        merged['Rct_last'] = np.nan
        merged['battery_impedance_last'] = np.nan
        merged['last_imp_flag'] = pd.NA
        merged['last_imp_anomaly'] = pd.NA
        merged['last_impedance_cycle_no'] = np.nan
    else:
        merged = pd.merge_asof(
            one_dis.sort_values('start_time'),
            one_imp.rename(columns={
                'Re': 'Re_last',
                'Rct': 'Rct_last',
                'battery_impedance': 'battery_impedance_last',
                'imp_flag': 'last_imp_flag',
                'imp_anomaly': 'last_imp_anomaly',
                'impedance_cycle_no': 'last_impedance_cycle_no'
            }),
            on='start_time',
            direction='backward'
        )
        merged['battery_id'] = battery_id

    aof_rows.append(merged)

cycle_master = pd.concat(aof_rows, ignore_index=True)

first_imp = imp_valid.groupby('battery_id')['start_time'].min().rename('first_imp_time')
cycle_master = cycle_master.merge(first_imp, on='battery_id', how='left')
cycle_master['impedance_available'] = (cycle_master['start_time'] >= cycle_master['first_imp_time']).fillna(False).astype(int)

print('전체 metadata 행 수        :', len(df))
print('discharge 행 수           :', len(df_discharge))
print('impedance 행 수           :', len(df_impedance))
print('cycle_master 행 수        :', len(cycle_master))
print('배터리 수                 :', cycle_master['battery_id'].nunique())
print('group 분포')
print(cycle_master['group'].value_counts().sort_index())


전체 metadata 행 수        : 7565
discharge 행 수           : 2794
impedance 행 수           : 1956
cycle_master 행 수        : 2794
배터리 수                 : 34
group 분포
group
A    636
B    112
C    160
D    591
E    141
F    403
G    288
H    100
I    363
Name: count, dtype: int64


## 5. Overview / Anomaly용 요약 테이블 생성

Tableau 초안에서 바로 쓰기 좋도록 **시트 목적별 CSV**로 나눠 만듭니다.


In [6]:

valid_cycle_master = cycle_master[cycle_master['cap_flag'] == 'valid'].copy()

latest_valid = (
    valid_cycle_master
    .sort_values(['battery_id', 'start_time'])
    .groupby('battery_id')
    .tail(1)
    .copy()
)

battery_flag_summary = (
    cycle_master.groupby('battery_id')
    .agg(
        group=('group', 'first'),
        analysis_role=('analysis_role', 'first'),
        total_discharge_rows=('battery_id', 'size'),
        hard_excluded_rows=('cap_exclude', 'sum'),
        low_capacity_anomaly_rows=('cap_anomaly', 'sum'),
        impedance_available_rows=('impedance_available', 'sum')
    )
    .reset_index()
)

imp_flag_summary = (
    df_impedance.groupby('battery_id')
    .agg(
        total_impedance_rows=('battery_id', 'size'),
        impedance_anomaly_rows=('imp_anomaly', 'sum'),
        impedance_excluded_rows=('imp_exclude', 'sum')
    )
    .reset_index()
)

battery_flag_summary = battery_flag_summary.merge(
    imp_flag_summary,
    on='battery_id',
    how='left'
).fillna(0)

latest_valid = latest_valid.merge(
    battery_flag_summary,
    on=['battery_id', 'group', 'analysis_role'],
    how='left'
)

latest_valid['eol_reached'] = (
    latest_valid['eol_cycle'].notna() &
    (latest_valid['discharge_cycle_raw'] >= latest_valid['eol_cycle'].fillna(np.inf))
)

latest_valid['dashboard_return_status'] = latest_valid.apply(classify_return_status, axis=1)

latest_valid['soh_status_band'] = pd.cut(
    latest_valid['SOH_nominal'],
    bins=[-np.inf, 70, 80, 90, np.inf],
    labels=['EOL 이하', '주의', '보통', '양호']
)

group_reference_rows = []

for group_code in sorted(df['group'].dropna().unique()):
    group_df = df[df['group'] == group_code].copy()
    batteries = sorted(group_df['battery_id'].dropna().unique().tolist())

    group_reference_rows.append({
        'group': group_code,
        'analysis_role': analysis_role_map.get(group_code),
        'end_reason': end_reason_map.get(group_code),
        'temperature_profile': test_temperature_profile_map.get(group_code),
        'load_profile': load_profile_map.get(group_code),
        'eol_rule_source': eol_rule_source_map.get(group_code),
        'battery_count': len(batteries),
        'battery_ids': ', '.join(batteries),
        'cutoff_voltage_summary': ', '.join([
            f"{battery_id}:{battery_protocol_map[battery_id]['cutoff_voltage']}V"
            for battery_id in batteries if battery_id in battery_protocol_map
        ])
    })

group_reference = pd.DataFrame(group_reference_rows)

group_summary = (
    latest_valid.groupby(['group', 'analysis_role'])
    .agg(
        battery_count=('battery_id', 'nunique'),
        avg_latest_capacity=('Capacity', 'mean'),
        avg_latest_soh_nominal=('SOH_nominal', 'mean'),
        avg_latest_soh_relative=('SOH_relative', 'mean'),
        avg_latest_rul=('RUL', 'mean'),
        avg_latest_rct=('Rct_last', 'mean'),
        eol_reached_count=('eol_reached', 'sum'),
        returnable_count=('dashboard_return_status', lambda s: (s == '귀환 가능').sum()),
        warning_count=('dashboard_return_status', lambda s: (s == '경고').sum()),
        nonreturnable_count=('dashboard_return_status', lambda s: (s == '귀환 불가').sum()),
        deferred_count=('dashboard_return_status', lambda s: (s == '판정보류').sum())
    )
    .reset_index()
)

group_summary['returnable_ratio'] = group_summary['returnable_count'] / group_summary['battery_count']

overview_kpi = pd.DataFrame([{
    'total_battery_count': int(latest_valid['battery_id'].nunique()),
    'total_group_count': int(latest_valid['group'].nunique()),
    'main_battery_count': int(latest_valid[latest_valid['analysis_role'] == 'main']['battery_id'].nunique()),
    'comparison_battery_count': int(latest_valid[latest_valid['analysis_role'] == 'comparison']['battery_id'].nunique()),
    'anomaly_battery_count': int(latest_valid[latest_valid['analysis_role'] == 'anomaly']['battery_id'].nunique()),
    'excluded_battery_count': int(latest_valid[latest_valid['analysis_role'] == 'excluded']['battery_id'].nunique()),
    'discharge_row_count': int(len(df_discharge)),
    'valid_discharge_row_count': int((df_discharge['cap_flag'] == 'valid').sum()),
    'impedance_row_count': int(len(df_impedance)),
    'avg_latest_soh_nominal': round(latest_valid['SOH_nominal'].mean(), 3),
    'avg_latest_soh_relative': round(latest_valid['SOH_relative'].mean(), 3),
    'avg_latest_rul_main': round(latest_valid[latest_valid['analysis_role'] == 'main']['RUL'].mean(), 3),
    'eol_reached_battery_count': int(latest_valid['eol_reached'].sum()),
    'returnable_battery_count': int((latest_valid['dashboard_return_status'] == '귀환 가능').sum()),
    'warning_battery_count': int((latest_valid['dashboard_return_status'] == '경고').sum()),
    'nonreturnable_battery_count': int((latest_valid['dashboard_return_status'] == '귀환 불가').sum()),
    'deferred_battery_count': int((latest_valid['dashboard_return_status'] == '판정보류').sum()),
    'returnable_ratio': round((latest_valid['dashboard_return_status'] == '귀환 가능').mean(), 4)
}])

overview_soh_band_summary = (
    latest_valid.groupby('soh_status_band', observed=False)
    .agg(
        battery_count=('battery_id', 'nunique'),
        avg_soh_nominal=('SOH_nominal', 'mean')
    )
    .reset_index()
)

overview_eol_treemap = latest_valid[
    ['battery_id', 'group', 'analysis_role', 'discharge_cycle_raw', 'eol_cycle', 'eol_soh_threshold', 'SOH_nominal']
].copy()

overview_eol_treemap['eol_status'] = np.where(
    overview_eol_treemap['eol_cycle'].notna(),
    'EOL 도달',
    'EOL 미도달'
)

overview_eol_treemap['size_value'] = np.where(
    overview_eol_treemap['eol_cycle'].notna(),
    overview_eol_treemap['eol_cycle'],
    overview_eol_treemap['discharge_cycle_raw']
)

overview_eol_treemap['cycle_gap_to_eol'] = (
    overview_eol_treemap['eol_cycle'] - overview_eol_treemap['discharge_cycle_raw']
)

overview_return_status = latest_valid[
    ['battery_id', 'group', 'analysis_role', 'SOH_nominal', 'eol_soh_threshold', 'dashboard_return_status', 'discharge_cycle_raw', 'RUL']
].copy()

battery_latest_export = latest_valid.rename(columns={
    'start_time': 'latest_start_time',
    'discharge_cycle_raw': 'latest_cycle_raw',
    'discharge_cycle_valid': 'latest_cycle_valid',
    'Capacity': 'latest_capacity_ah',
    'RUL': 'latest_rul'
})[[
    'battery_id', 'group', 'analysis_role', 'latest_start_time', 'latest_cycle_raw', 'latest_cycle_valid',
    'latest_capacity_ah', 'SOH_nominal', 'SOH_relative', 'latest_rul', 'eol_cycle', 'eol_soh_threshold',
    'eol_reached', 'Rct_last', 'Re_last', 'dashboard_return_status', 'soh_status_band',
    'hard_excluded_rows', 'low_capacity_anomaly_rows', 'impedance_anomaly_rows', 'total_discharge_rows',
    'total_impedance_rows', 'cutoff_voltage', 'test_temperature_profile', 'load_profile', 'end_reason'
]].copy()

cycle_trend_export = valid_cycle_master[[
    'battery_id', 'group', 'analysis_role', 'start_time', 'filename', 'discharge_cycle_raw', 'discharge_cycle_valid',
    'Capacity', 'SOH_nominal', 'SOH_relative', 'RUL', 'eol_cycle', 'eol_soh_threshold',
    'Rct_last', 'Re_last', 'battery_impedance_last', 'impedance_available', 'ambient_temperature',
    'cap_flag', 'cap_exclude', 'cap_anomaly', 'cutoff_voltage', 'test_temperature_profile', 'load_profile', 'end_reason'
]].copy()

anomaly_cycle_scatter = cycle_master[[
    'battery_id', 'group', 'analysis_role', 'start_time', 'filename',
    'discharge_cycle_raw', 'discharge_cycle_valid', 'Capacity', 'SOH_nominal',
    'SOH_relative', 'RUL', 'Rct_last', 'Re_last', 'battery_impedance_last',
    'ambient_temperature', 'cap_flag', 'cap_exclude', 'cap_anomaly', 'impedance_available'
]].copy()

anomaly_impedance_scatter = df_impedance[[
    'battery_id', 'group', 'analysis_role', 'start_time', 'filename',
    'impedance_cycle_no', 'Re', 'Rct', 'battery_impedance',
    'ambient_temperature', 'imp_flag', 'imp_exclude', 'imp_anomaly'
]].copy()

print('overview_kpi shape               :', overview_kpi.shape)
print('battery_latest_export shape      :', battery_latest_export.shape)
print('cycle_trend_export shape         :', cycle_trend_export.shape)
print('anomaly_cycle_scatter shape      :', anomaly_cycle_scatter.shape)
print('anomaly_impedance_scatter shape  :', anomaly_impedance_scatter.shape)


overview_kpi shape               : (1, 18)
battery_latest_export shape      : (34, 26)
cycle_trend_export shape         : (2553, 25)
anomaly_cycle_scatter shape      : (2794, 19)
anomaly_impedance_scatter shape  : (1956, 13)


## 6. ML용 테이블 생성 및 학습

ML 탭은 원본 notebook의 핵심 feature 구조를 그대로 씁니다.

### SOH 모델
- `discharge_cycle_valid`
- `Rct_last`
- `impedance_available`

### RUL 모델
- `discharge_cycle_raw`
- `SOH_nominal`
- `Rct_last`
- `impedance_available`

### 주의
- ML 결과는 `analysis_role == main`
- `rul_label_type == supervised`
- valid capacity 기준만 사용


In [7]:

ml_base_df = cycle_master[
    (cycle_master['analysis_role'] == 'main') &
    (cycle_master['cap_flag'] == 'valid') &
    (cycle_master['RUL'].notna()) &
    (cycle_master['rul_label_type'] == 'supervised')
].copy()

ml_base_df = ml_base_df.sort_values(['battery_id', 'start_time']).reset_index(drop=True)

soh_model_df = ml_base_df[
    ['battery_id', 'group', 'analysis_role', 'start_time', 'filename',
     'discharge_cycle_raw', 'discharge_cycle_valid',
     'SOH_nominal', 'Capacity', 'Rct_last', 'impedance_available']
].copy()

rul_model_df = ml_base_df[
    ['battery_id', 'group', 'analysis_role', 'start_time', 'filename',
     'discharge_cycle_raw', 'SOH_nominal', 'Capacity',
     'RUL', 'Rct_last', 'impedance_available']
].copy()

soh_feature_cols = ['discharge_cycle_valid', 'Rct_last', 'impedance_available']
rul_feature_cols = ['discharge_cycle_raw', 'SOH_nominal', 'Rct_last', 'impedance_available']

soh_xgb = run_single_split_regression(
    model_df=soh_model_df,
    feature_cols=soh_feature_cols,
    target_col='SOH_nominal',
    model_factory=make_xgb_regressor
)

rul_xgb = run_single_split_regression(
    model_df=rul_model_df,
    feature_cols=rul_feature_cols,
    target_col='RUL',
    model_factory=make_xgb_regressor
)

soh_full = fit_full_model(
    model_df=soh_model_df,
    feature_cols=soh_feature_cols,
    target_col='SOH_nominal',
    model_factory=make_xgb_regressor
)

rul_full = fit_full_model(
    model_df=rul_model_df,
    feature_cols=rul_feature_cols,
    target_col='RUL',
    model_factory=make_xgb_regressor
)

rul_lobo_detail, rul_lobo_pred = run_lobo(
    model_df=rul_model_df,
    feature_cols=rul_feature_cols,
    target_col='RUL',
    model_factory=make_xgb_regressor
)

print('ml_base_df shape:', ml_base_df.shape)
print('ml battery 수   :', ml_base_df['battery_id'].nunique())
print()
print('SOH metric')
print(soh_xgb['metric_df'])
print()
print('RUL metric')
print(rul_xgb['metric_df'])
print()
print('RUL LOBO')
print(rul_lobo_detail)


ml_base_df shape: (468, 39)
ml battery 수   : 3

SOH metric
     set     MAE    RMSE      R2
0  train  0.8281  1.1846  0.9878
1   test  2.2228  2.9034  0.9064

RUL metric
     set      MAE     RMSE      R2
0  train   1.4086   1.9417  0.9968
1   test  10.4256  12.7997  0.9037

RUL LOBO
  held_out_battery      MAE     RMSE      R2  row_count
0            B0005  10.4256  12.7997  0.9037        168
1            B0006   7.1563  10.5831  0.9143        168
2            B0018   8.3225  11.4736  0.8716        132


## 7. ML Tableau용 CSV 가공

초안에 맞춰 아래 CSV를 만듭니다.

1. 현재 predicted RUL  
2. actual vs predicted capacity trajectory  
3. 예측 신뢰도  
4. feature importance  


In [8]:

ml_model_metrics = pd.concat([
    soh_xgb['metric_df'].assign(target='SOH', model_name='XGBoost', feature_mode='baseline'),
    rul_xgb['metric_df'].assign(target='RUL', model_name='XGBoost', feature_mode='baseline')
], ignore_index=True)

ml_feature_importance = pd.concat([
    soh_full['importance_df'].assign(target='SOH', model_name='XGBoost', feature_mode='baseline'),
    rul_full['importance_df'].assign(target='RUL', model_name='XGBoost', feature_mode='baseline')
], ignore_index=True)

ml_feature_importance['rank_within_target'] = (
    ml_feature_importance.groupby('target')['importance']
    .rank(ascending=False, method='dense')
    .astype(int)
)

ml_capacity_trajectory = soh_xgb['pred_df'].copy()
ml_capacity_trajectory = ml_capacity_trajectory.rename(columns={
    'y_true': 'actual_soh',
    'y_pred': 'predicted_soh'
})
ml_capacity_trajectory['actual_capacity_ah'] = ml_capacity_trajectory['actual_soh'] / 100 * 2.0
ml_capacity_trajectory['predicted_capacity_ah'] = ml_capacity_trajectory['predicted_soh'] / 100 * 2.0

ml_rul_prediction_series = rul_xgb['pred_df'].copy()
ml_rul_prediction_series = ml_rul_prediction_series.rename(columns={
    'y_true': 'actual_rul',
    'y_pred': 'predicted_rul'
})
ml_rul_prediction_series['confidence_score'] = 1 / (1 + ml_rul_prediction_series['abs_error'])
ml_rul_prediction_series['confidence_band'] = pd.cut(
    ml_rul_prediction_series['abs_error'],
    bins=[-np.inf, 5, 10, 20, np.inf],
    labels=['높음', '중간', '낮음', '매우 낮음']
)

ml_current_rul = rul_full['pred_df'].copy()
ml_current_rul = ml_current_rul.rename(columns={
    'y_true': 'actual_rul',
    'y_pred': 'predicted_rul'
})
ml_current_rul = (
    ml_current_rul
    .sort_values(['battery_id', 'start_time'])
    .groupby('battery_id')
    .tail(1)
    .copy()
)
ml_current_rul['predicted_rul_round'] = ml_current_rul['predicted_rul'].round(2)

ml_error_by_battery = (
    ml_rul_prediction_series.groupby(['battery_id', 'set'])
    .agg(
        row_count=('battery_id', 'size'),
        mean_abs_error=('abs_error', 'mean'),
        rmse=('abs_error', lambda s: np.sqrt(np.mean(np.square(s)))),
        max_abs_error=('abs_error', 'max')
    )
    .reset_index()
    .round(4)
)

ml_error_band_summary = (
    ml_rul_prediction_series.groupby(['set', 'error_band'], observed=False)
    .size()
    .reset_index(name='row_count')
)

ml_lobo_detail = rul_lobo_detail.copy()

print('ml_model_metrics')
print(ml_model_metrics)
print()
print('ml_feature_importance')
print(ml_feature_importance)
print()
print('ml_current_rul')
print(ml_current_rul[['battery_id', 'predicted_rul_round', 'actual_rul']])


ml_model_metrics
     set      MAE     RMSE      R2 target model_name feature_mode
0  train   0.8281   1.1846  0.9878    SOH    XGBoost     baseline
1   test   2.2228   2.9034  0.9064    SOH    XGBoost     baseline
2  train   1.4086   1.9417  0.9968    RUL    XGBoost     baseline
3   test  10.4256  12.7997  0.9037    RUL    XGBoost     baseline

ml_feature_importance
                 feature  importance target model_name feature_mode  \
0  discharge_cycle_valid    0.871931    SOH    XGBoost     baseline   
1    impedance_available    0.065052    SOH    XGBoost     baseline   
2               Rct_last    0.063016    SOH    XGBoost     baseline   
3            SOH_nominal    0.824673    RUL    XGBoost     baseline   
4    discharge_cycle_raw    0.106595    RUL    XGBoost     baseline   
5    impedance_available    0.054788    RUL    XGBoost     baseline   
6               Rct_last    0.013944    RUL    XGBoost     baseline   

   rank_within_target  
0                   1  
1            

## 8. raw 파일 연결 점검 + raw discharge 상세 점 데이터 생성

이 섹션은 **있으면 좋은 보너스**입니다.

- `metadata.csv`만 있으면 overview / anomaly summary / ML summary는 생성 가능
- raw csv까지 있으면 `전압-용량`, `온도-용량` 같은 상세 anomaly 시트를 더 만들 수 있음
- raw 파일이 없으면 빈 CSV 또는 일부 행만 생성되고, 노트북은 계속 진행됨


In [9]:

raw_root_candidates = [
    metadata_path.parent,
    Path.cwd(),
    Path.cwd() / 'data',
    Path('/mnt/data'),
    Path('/mnt/data/data')
]

raw_root_candidates = unique_existing_paths(raw_root_candidates)

def find_raw_file(filename):
    for root in raw_root_candidates:
        one_path = root / str(filename)
        if one_path.exists():
            return one_path
    return None

raw_file_coverage = df[['filename', 'type', 'battery_id', 'start_time']].drop_duplicates().copy()
raw_file_coverage['found_path'] = raw_file_coverage['filename'].astype(str).apply(
    lambda x: str(find_raw_file(x)) if find_raw_file(x) else ''
)
raw_file_coverage['raw_file_found'] = raw_file_coverage['found_path'].ne('')

raw_discharge_rows = []

discharge_key_df = df_discharge[
    ['battery_id', 'group', 'analysis_role', 'start_time', 'filename', 'discharge_cycle_raw', 'Capacity', 'cap_flag']
].drop_duplicates().copy()

for _, row in discharge_key_df.iterrows():
    raw_path = find_raw_file(row['filename'])

    if raw_path is None:
        continue

    try:
        raw_df = pd.read_csv(raw_path)
    except Exception:
        continue

    needed_cols = {'Time', 'Voltage_measured', 'Temperature_measured'}
    if not needed_cols.issubset(set(raw_df.columns)):
        continue

    time_s = pd.to_numeric(raw_df['Time'], errors='coerce')
    voltage = pd.to_numeric(raw_df['Voltage_measured'], errors='coerce')
    temperature = pd.to_numeric(raw_df['Temperature_measured'], errors='coerce')

    if 'Current_load' in raw_df.columns:
        current_used = pd.to_numeric(raw_df['Current_load'], errors='coerce').abs()
    elif 'Current_measured' in raw_df.columns:
        current_used = pd.to_numeric(raw_df['Current_measured'], errors='coerce').abs()
    else:
        current_used = pd.Series(np.nan, index=raw_df.index)

    point_df = pd.DataFrame({
        'battery_id': row['battery_id'],
        'group': row['group'],
        'analysis_role': row['analysis_role'],
        'start_time': row['start_time'],
        'filename': row['filename'],
        'discharge_cycle_raw': row['discharge_cycle_raw'],
        'event_capacity_ah': row['Capacity'],
        'cap_flag': row['cap_flag'],
        'point_index': np.arange(len(raw_df)),
        'time_sec': time_s,
        'voltage_measured': voltage,
        'temperature_measured': temperature,
        'current_used_abs': current_used
    })

    point_df = point_df.dropna(subset=['time_sec']).sort_values('time_sec').reset_index(drop=True)
    point_df['delta_time_sec'] = point_df['time_sec'].diff().clip(lower=0).fillna(0)
    point_df['delta_ah'] = point_df['current_used_abs'] * point_df['delta_time_sec'] / 3600
    point_df['cumulative_ah_raw'] = point_df['delta_ah'].fillna(0).cumsum()

    final_cumulative_ah = point_df['cumulative_ah_raw'].max()

    if pd.notna(row['Capacity']) and row['Capacity'] > 0 and pd.notna(final_cumulative_ah) and final_cumulative_ah > 0:
        point_df['capacity_axis_ah'] = point_df['cumulative_ah_raw'] / final_cumulative_ah * row['Capacity']
    else:
        point_df['capacity_axis_ah'] = point_df['cumulative_ah_raw']

    point_df['capacity_axis_pct'] = np.where(
        pd.notna(point_df['event_capacity_ah']) & (point_df['event_capacity_ah'] > 0),
        point_df['capacity_axis_ah'] / point_df['event_capacity_ah'] * 100,
        np.nan
    )

    raw_discharge_rows.append(point_df)

if len(raw_discharge_rows) > 0:
    raw_discharge_points = pd.concat(raw_discharge_rows, ignore_index=True)
else:
    raw_discharge_points = pd.DataFrame(columns=[
        'battery_id', 'group', 'analysis_role', 'start_time', 'filename',
        'discharge_cycle_raw', 'event_capacity_ah', 'cap_flag', 'point_index',
        'time_sec', 'voltage_measured', 'temperature_measured', 'current_used_abs',
        'delta_time_sec', 'delta_ah', 'cumulative_ah_raw', 'capacity_axis_ah', 'capacity_axis_pct'
    ])

print('raw_root_candidates')
for one_root in raw_root_candidates:
    print('-', one_root)

print()
print('raw_file_coverage found 비율 :', raw_file_coverage['raw_file_found'].mean())
print('raw_discharge_points shape    :', raw_discharge_points.shape)


raw_root_candidates
- C:\Users\박정수\OneDrive\바탕 화면\spartal_python\16_project\project_lithium_battery_analysis_16team\data
- C:\Users\박정수\OneDrive\바탕 화면\spartal_python\16_project\project_lithium_battery_analysis_16team

raw_file_coverage found 비율 : 1.0
raw_discharge_points shape    : (770070, 18)


## 9. Tableau CSV 저장

아래 셀을 실행하면 `tableau_csv_outputs` 폴더에 CSV가 저장됩니다.  
같은 폴더를 ZIP으로도 묶습니다.


In [10]:

output_dir_candidates = [
    Path.cwd() / 'tableau_csv_outputs',
    Path('/mnt/data/tableau_csv_outputs')
]

output_dir = None
for one_dir in output_dir_candidates:
    try:
        one_dir.mkdir(parents=True, exist_ok=True)
        test_file = one_dir / '__write_test__.tmp'
        test_file.write_text('ok', encoding='utf-8')
        test_file.unlink()
        output_dir = one_dir
        break
    except Exception:
        continue

if output_dir is None:
    raise PermissionError('CSV를 저장할 수 있는 폴더를 찾지 못했습니다.')

exports = {
    'tableau_overview_kpi.csv': overview_kpi,
    'tableau_overview_group_reference.csv': group_reference,
    'tableau_overview_group_summary.csv': group_summary.round(4),
    'tableau_overview_battery_latest.csv': battery_latest_export.round(4),
    'tableau_overview_cycle_trend.csv': cycle_trend_export.round(4),
    'tableau_overview_eol_treemap.csv': overview_eol_treemap.round(4),
    'tableau_overview_soh_band_summary.csv': overview_soh_band_summary.round(4),
    'tableau_overview_return_status.csv': overview_return_status.round(4),

    'tableau_anomaly_cycle_scatter.csv': anomaly_cycle_scatter.round(4),
    'tableau_anomaly_impedance_scatter.csv': anomaly_impedance_scatter.round(4),
    'tableau_anomaly_raw_discharge_points.csv': raw_discharge_points.round(6) if len(raw_discharge_points) > 0 else raw_discharge_points,

    'tableau_ml_model_metrics.csv': ml_model_metrics.round(4),
    'tableau_ml_feature_importance.csv': ml_feature_importance.round(6),
    'tableau_ml_capacity_trajectory.csv': ml_capacity_trajectory.round(6),
    'tableau_ml_rul_prediction_series.csv': ml_rul_prediction_series.round(6),
    'tableau_ml_current_rul.csv': ml_current_rul.round(6),
    'tableau_ml_error_by_battery.csv': ml_error_by_battery.round(6),
    'tableau_ml_error_band_summary.csv': ml_error_band_summary,
    'tableau_ml_lobo_detail.csv': ml_lobo_detail.round(6),

    'tableau_raw_file_coverage.csv': raw_file_coverage
}

for file_name, export_df in exports.items():
    export_df.to_csv(output_dir / file_name, index=False)

manifest_df = pd.DataFrame([
    {
        'file_name': file_name,
        'row_count': len(export_df),
        'column_count': len(export_df.columns)
    }
    for file_name, export_df in exports.items()
])

manifest_df = manifest_df.sort_values('file_name').reset_index(drop=True)
manifest_df.to_csv(output_dir / 'tableau_export_manifest.csv', index=False)

zip_path = output_dir.parent / 'tableau_csv_outputs.zip'
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for one_file in sorted(output_dir.glob('*.csv')):
        zf.write(one_file, arcname=one_file.name)

print('output_dir :', output_dir)
print('zip_path   :', zip_path)
print()
print(manifest_df)


output_dir : c:\Users\박정수\OneDrive\바탕 화면\spartal_python\16_project\project_lithium_battery_analysis_16team\tableau_csv_outputs
zip_path   : c:\Users\박정수\OneDrive\바탕 화면\spartal_python\16_project\project_lithium_battery_analysis_16team\tableau_csv_outputs.zip

                                   file_name  row_count  column_count
0          tableau_anomaly_cycle_scatter.csv       2794            19
1      tableau_anomaly_impedance_scatter.csv       1956            13
2   tableau_anomaly_raw_discharge_points.csv     770070            18
3         tableau_ml_capacity_trajectory.csv        468            18
4                 tableau_ml_current_rul.csv          3            15
5          tableau_ml_error_band_summary.csv          8             3
6            tableau_ml_error_by_battery.csv          3             6
7          tableau_ml_feature_importance.csv          7             6
8                 tableau_ml_lobo_detail.csv          3             5
9               tableau_ml_model_metrics.

## 10. 빠른 확인

이 셀은 저장된 결과를 마지막으로 점검하는 용도입니다.


In [11]:

print('overview KPI')
display(pd.read_csv(output_dir / 'tableau_overview_kpi.csv'))

print('\nML metrics')
display(pd.read_csv(output_dir / 'tableau_ml_model_metrics.csv'))

print('\nmanifest')
display(pd.read_csv(output_dir / 'tableau_export_manifest.csv'))


overview KPI


,total_battery_count,total_group_count,main_battery_count,comparison_battery_count,anomaly_battery_count,excluded_battery_count,discharge_row_count,valid_discharge_row_count,impedance_row_count,avg_latest_soh_nominal,avg_latest_soh_relative,avg_latest_rul_main,eol_reached_battery_count,returnable_battery_count,warning_battery_count,nonreturnable_battery_count,deferred_battery_count,returnable_ratio
0,34,9,4,4,16,10,2794,2553,1956,63.309,88.556,0.0,22,8,1,21,4,0.2353



ML metrics


,set,MAE,RMSE,R2,target,model_name,feature_mode
0,train,0.8281,1.1846,0.9878,SOH,XGBoost,baseline
1,test,2.2228,2.9034,0.9064,SOH,XGBoost,baseline
2,train,1.4086,1.9417,0.9968,RUL,XGBoost,baseline
3,test,10.4256,12.7997,0.9037,RUL,XGBoost,baseline



manifest


,file_name,row_count,column_count
0,tableau_anomaly_cycle_scatter.csv,2794,19
1,tableau_anomaly_impedance_scatter.csv,1956,13
2,tableau_anomaly_raw_discharge_points.csv,770070,18
3,tableau_ml_capacity_trajectory.csv,468,18
4,tableau_ml_current_rul.csv,3,15
5,tableau_ml_error_band_summary.csv,8,3
6,tableau_ml_error_by_battery.csv,3,6
7,tableau_ml_feature_importance.csv,7,6
8,tableau_ml_lobo_detail.csv,3,5
9,tableau_ml_model_metrics.csv,4,7
